# Factor Pricing ML — Exploration

Pipeline complet : données -> baseline OLS -> modèles ML -> comparaison hors échantillon -> backtest.

In [2]:
import sys
sys.path.append('../src')

import pandas as pd
import matplotlib.pyplot as plt

from data_loader import load_fama_french_5factors, load_asset_returns
#from models import fit_ols_baseline, fit_ml_models, evaluate_models, time_series_split
from backtest import compare_to_buy_and_hold

%matplotlib inline

## 1. Chargement des données

In [ ]:
ff5 = load_fama_french_5factors()
ff5.tail()

In [ ]:
# Actif test : S&P 500 (à remplacer par un portefeuille trié si tu veux aller plus loin)
asset = load_asset_returns(["^GSPC"], start="2000-01-01")
asset.columns = ["SP500"]
asset.tail()

## 2. Préparation des données (aligner les dates, calculer le rendement excédentaire)

In [ ]:
df = ff5.join(asset, how="inner")
df["excess_ret"] = df["SP500"] - df["RF"]
df = df.dropna()

X = df[["Mkt-RF", "SMB", "HML", "RMW", "CMA"]]
y = df["excess_ret"]

X_train, X_test, y_train, y_test = time_series_split(X, y, test_size=0.2)
print(f"Train: {len(X_train)} obs, Test: {len(X_test)} obs")

## 3. Baseline OLS

In [ ]:
ols_model = fit_ols_baseline(y_train, X_train)
print(ols_model.summary())

## 4. Modèles ML

In [ ]:
ml_models = fit_ml_models(X_train, y_train)
results = evaluate_models(ml_models, X_test, y_test, ols_model=ols_model, X_test_ols=X_test)
results

## 5. Backtest : la meilleure stratégie ML bat-elle le buy-and-hold ?

In [ ]:
best_model_name = results.iloc[0]["model"]
if best_model_name in ml_models:
    best_preds = pd.Series(ml_models[best_model_name].predict(X_test), index=X_test.index)
    summary = compare_to_buy_and_hold(best_preds, y_test)
    display(summary)
else:
    print("Le meilleur modèle est l'OLS — refais tourner avec ses prédictions si besoin.")

## Prochaines étapes

- Étendre à d'autres actifs / portefeuilles triés (taille, valeur)
- Ajouter du feature engineering (rendements décalés, volatilité réalisée)
- Analyse d'importance des facteurs (SHAP) pour interpréter les modèles ML
- Tester la robustesse sur différentes périodes (crise 2008, COVID, etc.)